# 1 Import libraries

In [1]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from pathlib import Path
import re
import time
from datetime import datetime

# 2. Define settings

In [2]:
# -----------------------------
# Settings
# -----------------------------

BASE_URL = 'https://www.fcc.gov/documents?field_edoc_doctype_target_id%5B%5D=92&field_released_date_value=&exposed_from_date=&exposed_to_date='

FCC_HOME = 'https://www.fcc.gov'

PAGE_START = 0
PAGE_END = 4
OUTPUT_FOLDER = Path('fcc_documents')

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (academic research)',
    'Accept-Language': 'en-US',
}

# 3. Define helper functions

In [3]:
def create_session(headers=HEADERS):
    '''
    Create a requests session with browser-like headers.
    '''
    session = requests.Session()
    session.headers.update(headers)
    return session


def get_soup(session, url, timeout=30):
    '''
    Download a web page and return a BeautifulSoup object.
    The code stops if the request fails.
    '''
    response = session.get(url, timeout=timeout)
    response.raise_for_status()
    return BeautifulSoup(response.text, 'html.parser')


def build_search_url(page):
    '''
    Build the FCC search result URL for a given page number.
    '''
    return f'{BASE_URL}&page={page}'


def extract_document_links_from_soup(soup):
    '''
    Extract document titles and document page links from one FCC search result page.
    '''
    documents = []

    for a in soup.select('div.headline-title a.title'):
        title = a.get_text(strip=True)
        href = a.get('href')

        if not href:
            continue

        link = urljoin(FCC_HOME, href)

        documents.append({
            'title': title,
            'link': link
        })

    return documents


def collect_document_links(session, start_page=PAGE_START, end_page=PAGE_END, delay=1):
    '''
    Go through FCC search result pages and collect document titles and links.
    '''
    document_links = []

    for page in range(start_page, end_page + 1):
        print('=' * 60)
        print(f'Search page {page}')

        page_url = build_search_url(page)

        try:
            soup = get_soup(session, page_url)
            page_documents = extract_document_links_from_soup(soup)

            document_links.extend(page_documents)

            for doc in page_documents:
                print(doc['title'], '>>>', doc['link'])

            print(f'Documents found on page {page}: {len(page_documents)}')

        except requests.RequestException as e:
            print(f'>>> Failed to collect page {page}: {e}')

        time.sleep(delay)

    print('\nTotal number of document links:', len(document_links))
    return document_links


def clean_filename(title):
    '''
    Convert a document title into a safe file name.
    Windows does not allow certain special characters in file names.
    '''
    clean_title = re.sub(r'[\\/*?:"<>|]', '', title)
    clean_title = re.sub(r'\s+', '_', clean_title)
    clean_title = clean_title.strip('_')

    if not clean_title:
        clean_title = 'untitled'

    return clean_title

def format_date_for_filename(date_text):
    '''
    Convert FCC date text like "Apr 20, 2026" into "260420".
    '''
    date_text = date_text.strip()

    for date_format in ['%b %d, %Y', '%B %d, %Y']:
        try:
            date_object = datetime.strptime(date_text, date_format)
            return date_object.strftime('%y%m%d')
        except ValueError:
            continue

    return 'unknown_date'


def find_txt_url_and_issued_date(session, document_url):
    '''
    Open an FCC document page and find:
    1. the .txt file link
    2. the issued date
    '''
    soup = get_soup(session, document_url)

    txt_tag = soup.select_one('a.document-link.txt')

    if txt_tag is None:
        txt_url = None
    else:
        href = txt_tag.get('href')
        txt_url = urljoin(FCC_HOME, href) if href else None

    page_text = soup.get_text(' ', strip=True)

    date_match = re.search(
        r'Issued On\s*:\s*([A-Za-z]{3,9}\s+\d{1,2},\s+\d{4})',
        page_text
    )

    if date_match:
        issued_date = format_date_for_filename(date_match.group(1))
    else:
        issued_date = 'unknown_date'

    return txt_url, issued_date


def save_txt_file(session, txt_url, file_path, timeout=30):
    '''
    Download one .txt file and save it to the selected file path.
    '''
    response = session.get(txt_url, timeout=timeout)
    response.raise_for_status()

    file_path.write_text(response.text, encoding='utf-8')


def download_txt_documents(session, document_links, output_folder=OUTPUT_FOLDER, delay=1):
    '''
    For each FCC document page:
    1. find the .txt file link,
    2. download the .txt file,
    3. save it in the output folder.
    '''
    output_folder.mkdir(exist_ok=True)

    downloaded_files = []
    skipped_documents = []
    failed_documents = []

    for i, doc in enumerate(document_links, start=1):
        print('=' * 60)
        print(f"Document_{i}: {doc['title']}")

        try:
            txt_url, issued_date = find_txt_url_and_issued_date(session, doc['link'])

            if txt_url is None:
                print('>>> No text file found.')
                skipped_documents.append(doc)
                continue
            
            clean_title = clean_filename(doc['title'])
            file_path = output_folder / f'{issued_date}_{clean_title}.txt'
            
            save_txt_file(session, txt_url, file_path)

            downloaded_files.append(file_path)
            print('Saved:', file_path)

        except requests.RequestException as e:
            print(f'>>> Failed to download document {i}: {e}')
            failed_documents.append(doc)

        time.sleep(delay)

    print('\nDownload summary')
    print('Downloaded:', len(downloaded_files))
    print('Skipped because no .txt file was found:', len(skipped_documents))
    print('Failed:', len(failed_documents))

    return downloaded_files, skipped_documents, failed_documents

# 4. Collect document links from FCC search result pages

In [4]:
# Create a session
session = create_session()

# Collect document page links from pages 0 to 4
document_links = collect_document_links(
    session=session,
    start_page=PAGE_START,
    end_page=PAGE_END,
    delay=1
)

Search page 0
Trusty NAB Show Remarks >>> https://www.fcc.gov/document/trusty-nab-show-remarks
Trusty WTA Spring Educational Forum Remarks >>> https://www.fcc.gov/document/trusty-wta-spring-educational-forum-remarks
Commissioner Trusty Remarks at CLDP in Costa Rica >>> https://www.fcc.gov/document/commissioner-trusty-remarks-cldp-costa-rica
Commissioner Trusty Speaks at National Urban League >>> https://www.fcc.gov/document/commissioner-trusty-speaks-national-urban-league
Trusty Fiber Broadband Association Summit Remarks >>> https://www.fcc.gov/document/trusty-fiber-broadband-association-summit-remarks
Trusty State of the Net Keynote >>> https://www.fcc.gov/document/trusty-state-net-keynote
Gomez on Media Consolidation at SOTN >>> https://www.fcc.gov/document/gomez-media-consolidation-sotn
Trusty Incompas Policy Summit Keynote >>> https://www.fcc.gov/document/trusty-incompas-policy-summit-keynote
Trusty Hudson Institute Remarks >>> https://www.fcc.gov/document/trusty-hudson-institute-r

# 5. Download the `.txt` files

In [5]:
# Download .txt files from the collected document links
downloaded_files, skipped_documents, failed_documents = download_txt_documents(
    session=session,
    document_links=document_links,
    output_folder=OUTPUT_FOLDER,
    delay=1
)

Document_1: Trusty NAB Show Remarks
Saved: fcc_documents\260420_Trusty_NAB_Show_Remarks.txt
Document_2: Trusty WTA Spring Educational Forum Remarks
Saved: fcc_documents\260413_Trusty_WTA_Spring_Educational_Forum_Remarks.txt
Document_3: Commissioner Trusty Remarks at CLDP in Costa Rica
Saved: fcc_documents\260318_Commissioner_Trusty_Remarks_at_CLDP_in_Costa_Rica.txt
Document_4: Commissioner Trusty Speaks at National Urban League
Saved: fcc_documents\260318_Commissioner_Trusty_Speaks_at_National_Urban_League.txt
Document_5: Trusty Fiber Broadband Association Summit Remarks
Saved: fcc_documents\260224_Trusty_Fiber_Broadband_Association_Summit_Remarks.txt
Document_6: Trusty State of the Net Keynote
Saved: fcc_documents\260209_Trusty_State_of_the_Net_Keynote.txt
Document_7: Gomez on Media Consolidation at SOTN
Saved: fcc_documents\260209_Gomez_on_Media_Consolidation_at_SOTN.txt
Document_8: Trusty Incompas Policy Summit Keynote
Saved: fcc_documents\260204_Trusty_Incompas_Policy_Summit_Keynot